# Environment Setup Workbench

Use this notebook to create or refresh the `tatva` environment, install JAX and the local package, and verify that the runtime is ready.

This notebook is designed to work both on the local repository and on a copied repository such as the T7 drive.


## 1. Path Detection and Helpers

Run this cell first. It tries to auto-detect the repository root and defines a small command runner.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path


def detect_repo_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent])
    candidates.extend([
        Path('/Users/gausschang/Documents/University/Master/tatva'),
        Path('/Volumes/Gauss-T7/tatva'),
    ])
    for root in candidates:
        if (root / 'Velocity-weakening' / 'src' / 'velocity_weakening_tatva.py').exists() and (root / 'tatva' / 'legacy_velocity_weakening.py').exists():
            return root
    raise FileNotFoundError('Could not auto-detect the tatva repository root. Set REPO_ROOT manually in this cell.')


REPO_ROOT = detect_repo_root()
SRC_ROOT = REPO_ROOT / 'Velocity-weakening' / 'src'
CONDA_EXE = shutil.which('conda')

if CONDA_EXE is None:
    raise RuntimeError('`conda` was not found on PATH. Install Conda/Miniconda first or open Jupyter from a shell where `conda` is available.')


def run(cmd: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd or REPO_ROOT),
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {cmd}')
    return completed


print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT  =', SRC_ROOT)
print('CONDA_EXE =', CONDA_EXE)


## 2. Setup Configuration

Edit these values if needed. For a standard CPU setup, the defaults are usually fine.


In [ ]:
ENV_NAME = 'tatva'
PYTHON_VERSION = '3.12'
CREATE_ENV_IF_MISSING = True
FORCE_PIP_UPGRADE = True
INSTALL_EDITABLE_TATVA = True
INSTALL_CPU_JAX = True
INSTALL_OPTIONAL_NOTEBOOK_PACKAGES = True
REGISTER_IPYKERNEL = True

# Keep GPU plugin optional. CPU JAX is the default stable choice for this project.
INSTALL_JAX_METAL = False

NOTEBOOK_PACKAGES = [
    'matplotlib',
    'h5py',
    'pytest',
    'ipykernel',
    'pandas',
]

print(json.dumps({
    'ENV_NAME': ENV_NAME,
    'PYTHON_VERSION': PYTHON_VERSION,
    'CREATE_ENV_IF_MISSING': CREATE_ENV_IF_MISSING,
    'INSTALL_CPU_JAX': INSTALL_CPU_JAX,
    'INSTALL_JAX_METAL': INSTALL_JAX_METAL,
    'INSTALL_EDITABLE_TATVA': INSTALL_EDITABLE_TATVA,
}, indent=2))


## 3. Create the Conda Environment If Needed

This cell checks whether the named environment exists. If not, it creates it.


In [ ]:
env_list = run([CONDA_EXE, 'env', 'list', '--json'])
envs = json.loads(env_list.stdout).get('envs', [])
env_exists = any(Path(path).name == ENV_NAME for path in envs)
print('env_exists =', env_exists)

if not env_exists:
    if not CREATE_ENV_IF_MISSING:
        raise RuntimeError(f'Conda environment `{ENV_NAME}` does not exist and CREATE_ENV_IF_MISSING is False.')
    run([CONDA_EXE, 'create', '-n', ENV_NAME, f'python={PYTHON_VERSION}', '-y'])
else:
    print(f'Conda environment `{ENV_NAME}` already exists.')


## 4. Install or Refresh Packages

This cell upgrades pip tooling, installs CPU JAX by default, installs the local `tatva` package in editable mode, and adds notebook utilities.


In [ ]:
def conda_run_python(args: list[str]) -> subprocess.CompletedProcess[str]:
    return run([CONDA_EXE, 'run', '-n', ENV_NAME, 'python', *args])


if FORCE_PIP_UPGRADE:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

if INSTALL_CPU_JAX:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'jax', 'jaxlib'])

if INSTALL_JAX_METAL:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'jax-metal'])

if INSTALL_EDITABLE_TATVA:
    conda_run_python(['-m', 'pip', 'install', '-e', str(REPO_ROOT)])

if INSTALL_OPTIONAL_NOTEBOOK_PACKAGES and NOTEBOOK_PACKAGES:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', *NOTEBOOK_PACKAGES])

if REGISTER_IPYKERNEL:
    conda_run_python(['-m', 'ipykernel', 'install', '--user', '--name', ENV_NAME, '--display-name', f'Python ({ENV_NAME})'])


## 5. Verify the Environment

This cell checks the package versions, imports `tatva`, and prints the available JAX devices.


In [ ]:
verify_script = r'''
import json
import importlib.metadata as md
from pathlib import Path

import jax
import h5py
import matplotlib
import tatva

root = Path(r"''' + str(REPO_ROOT) + '''")
payload = {
    'repo_root': str(root),
    'jax_version': md.version('jax'),
    'jaxlib_version': md.version('jaxlib'),
    'matplotlib_version': md.version('matplotlib'),
    'h5py_version': md.version('h5py'),
    'tatva_import_ok': hasattr(tatva, '__file__'),
    'tatva_module': str(getattr(tatva, '__file__', 'unknown')),
    'backend': jax.default_backend(),
    'devices': [str(d) for d in jax.devices()],
}
print(json.dumps(payload, indent=2))
'''

conda_run_python(['-c', verify_script])


## 6. Optional Smoke Test

Turn this on if you want to run the repository smoke tests immediately after setup.


In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    run([
        CONDA_EXE,
        'run',
        '-n',
        ENV_NAME,
        'pytest',
        str(REPO_ROOT / 'tests' / 'test_legacy_velocity_weakening.py'),
        '-q',
    ])
else:
    print('Set RUN_SMOKE_TEST = True if you want to run pytest now.')
